# DriveGuard - Milestone 3 HPO (Kaggle GPU)

Optuna hyperparameter search on the three tied contenders (LightGBM, RandomForest, CatBoost)
on the rolling feature set. Objective = PR-AUC on a validation subsample; final tuned models
scored on the full held-out test (2025-Q3).

**Settings (right panel):** Add data `driveguard-backblaze-interim`; Accelerator = GPU;
Internet = On; Secret `GITHUB_TOKEN` (repo read).

Expect ~3-5 h total. Lower the trial counts in the HPO cell if you want it shorter.

In [ ]:
# 1. Clone repo
import os, subprocess
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
URL = f'https://{token}@github.com/keerthirevanth/driveguard-predictive-maintenance.git'
REPO = '/kaggle/working/driveguard-predictive-maintenance'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', URL, REPO], check=True)
import sys; sys.path.insert(0, f'{REPO}/src')
print('cloned:', os.path.exists(REPO))

In [ ]:
# 2. Install deps
!pip install -q polars pyarrow mlflow optuna catboost 2>/dev/null
print('deps installed')

In [ ]:
# 3. Link dataset (recursive + clears stale links)
import os, glob
os.makedirs(f'{REPO}/data/interim', exist_ok=True)
os.makedirs(f'{REPO}/data/processed', exist_ok=True)
qf = glob.glob('/kaggle/input/**/data_*.parquet', recursive=True)
sf = glob.glob('/kaggle/input/**/drive_summary.parquet', recursive=True)
assert qf and sf, 'dataset not attached?'
def link(src, dst):
    if os.path.lexists(dst): os.remove(dst)
    os.symlink(src, dst)
for f in qf: link(f, f"{REPO}/data/interim/{os.path.basename(f)}")
link(sf[0], f'{REPO}/data/processed/drive_summary.parquet')
print('interim:', sorted(os.listdir(f'{REPO}/data/interim')))

In [ ]:
# 4. Build rolling features (N=30)
from pathlib import Path
from driveguard.config import load_config
from driveguard.features.rolling import make_rolling_dataset
ROOT = Path(REPO); cfg = load_config(f'{REPO}/config/config.yaml'); N = 30
make_rolling_dataset(cfg, ROOT, N)
print('rolling features built')

In [ ]:
# 5. Optuna HPO on the three contenders (adjust trials to taste)
import json
from driveguard.models.hpo import run as run_hpo
FDIR = f'{REPO}/data/processed/features_rolling_N{N}'
TRIALS = {'lightgbm': 50, 'random_forest': 15, 'catboost': 15}
results = run_hpo(FDIR, ['lightgbm', 'random_forest', 'catboost'], cfg,
                  trials=TRIALS, mlflow_uri='/kaggle/working/mlruns')
json.dump(results, open(f'/kaggle/working/hpo_rolling_N{N}.json', 'w'), indent=2)
print('HPO complete')

In [ ]:
# 6. Tuned leaderboard vs untuned best (0.1448)
import pandas as pd
rows = [{'model': r['model'], 'val_pr_auc': round(r['best_val_pr_auc'], 4),
         'test_pr_auc': round(r['test']['pr_auc'], 4),
         'test_roc_auc': round(r['test']['roc_auc'], 3),
         'test_recall@1%fpr': round((r['test']['recall_at_fpr_1pct']['recall'] or 0), 3)}
        for r in results]
print('untuned best (rolling): random_forest PR-AUC 0.1448')
pd.DataFrame(rows).sort_values('test_pr_auc', ascending=False)

In [ ]:
# 7. Best params + package artifacts
for r in results:
    print(r['model'], '->', r['best_params'])
import shutil, os
if os.path.isdir('/kaggle/working/mlruns'):
    shutil.make_archive('/kaggle/working/mlruns_export', 'zip',
                        root_dir='/kaggle/working', base_dir='mlruns')
!ls -lh /kaggle/working/*.json